In [ ]:
import sys
import math
import matplotlib.pyplot as plt
import pandas as pd
import heapq


In [ ]:
def ler_tsp_arquivo(caminho_arquivo):
    coords = []
    lendo = False

    with open(caminho_arquivo, 'r') as f:
        for linha in f:
            linha = linha.strip()

            if linha == "NODE_COORD_SECTION":
                lendo = True
                continue

            if linha == "EOF":
                break

            if lendo:
                partes = linha.split()
                id = int(partes[0])
                x = float(partes[1])
                y = float(partes[2])
                coords.append((id, x, y))

    return coords

def calcular_distancia(cidade1, cidade2):
    _, xi, yi = cidade1
    _, xj, yj = cidade2

    dist = math.sqrt((xi - xj)**2 + (yi - yj)**2)
    return math.floor(0.5 + dist)

def criar_matriz_distancias(coords):
    n = len(coords)

    matriz = [[0 for _ in range(n)] for _ in range(n)]

    for i in range(n):
        for j in range(n):
            if i != j:
                matriz[i][j] = calcular_distancia(coords[i], coords[j])

    return matriz


In [ ]:
#plotar grafico
def preparar_coords(coords):
    return {int(id_): (x, y) for id_, x, y in coords}


def configurar_plot(titulo):
    plt.figure(figsize=(18, 8))
    plt.title(titulo, fontsize=16)
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.grid(True, alpha=0.3)


def desenhar_vertices(coords_dict):
    for vertice, (x, y) in coords_dict.items():
        plt.scatter(x, y, s=90, zorder=3)
        plt.text(
            x,
            y,
            str(vertice),
            fontsize=12,
            verticalalignment="bottom",
            horizontalalignment="right",
            zorder=4
        )


def desenhar_aresta(coords_dict, u, v, peso=None, mostrar_peso=True, linewidth=2, alpha=1):
    x1, y1 = coords_dict[u]
    x2, y2 = coords_dict[v]

    plt.plot(
        [x1, x2],
        [y1, y2],
        linewidth=linewidth,
        alpha=alpha,
        zorder=1
    )

    if mostrar_peso and peso is not None:
        xm = (x1 + x2) / 2
        ym = (y1 + y2) / 2

        plt.text(
            xm,
            ym,
            str(peso),
            fontsize=9,
            zorder=2
        )


def plotar_agm(coords, agm, titulo, custo):
    coords_dict = preparar_coords(coords)

    configurar_plot(f"{titulo} - Custo: {custo:.2f}")

    for peso, vertice, pai in agm:
        desenhar_aresta(
            coords_dict,
            pai,
            vertice,
            peso=peso,
            mostrar_peso=True,
            linewidth=2,
            alpha=1
        )

    desenhar_vertices(coords_dict)
    plt.show()


def plotar_grafo(coords, matriz):
    coords_dict = preparar_coords(coords)
    n = len(coords)

    configurar_plot("Grafo Completo")

    for i in range(1, n + 1):
        for j in range(i + 1, n + 1):
            peso = matriz[i - 1][j - 1]

            if peso != 0:
                desenhar_aresta(
                    coords_dict,
                    i,
                    j,
                    peso=peso,
                    mostrar_peso=True,
                    linewidth=1,
                    alpha=0.25
                )

    desenhar_vertices(coords_dict)
    plt.show()


def plotar_ciclo(coords, ciclo, custo, matriz, titulo="Ciclo Hamiltoniano"):
    coords_dict = preparar_coords(coords)

    configurar_plot(f"{titulo} - Custo: {custo:.2f}")

    for i in range(len(ciclo) - 1):
        u = ciclo[i]
        v = ciclo[i + 1]

        peso = matriz[u - 1][v - 1]

        desenhar_aresta(
            coords_dict,
            u,
            v,
            peso=peso,
            mostrar_peso=True,
            linewidth=2,
            alpha=1
        )

    desenhar_vertices(coords_dict)
    plt.show()


def plotar_tour(coords, tour, titulo="Tour"):
    coords_dict = preparar_coords(coords)

    configurar_plot(titulo)

    for i in range(len(tour) - 1):
        u = tour[i]
        v = tour[i + 1]

        desenhar_aresta(
            coords_dict,
            u,
            v,
            peso=None,
            mostrar_peso=False,
            linewidth=2,
            alpha=1
        )

    desenhar_vertices(coords_dict)
    plt.show()

In [ ]:
def kruskal_agm(grafo):
    n = len(grafo)

    agm = []
    tupla = []
    custo_total = 0

    for i in range(n):
        for j in range(i + 1, n):
            peso = grafo[i][j]

            if peso != 0:
                tupla.append((peso, i, j))

    tupla_ordenada = sorted(tupla, key=lambda x: x[0])

    pai = list(range(n))

    def find(v):
        if pai[v] != v:
            pai[v] = find(pai[v])
        return pai[v]

    def union(origem, destino):
        raiz_origem = find(origem)
        raiz_destino = find(destino)

        if raiz_origem != raiz_destino:
            pai[raiz_destino] = raiz_origem
            return True

        return False

    for peso, origem, destino in tupla_ordenada:
        if union(origem, destino):
            agm.append((peso, origem + 1, destino + 1))
            custo_total += peso

        if len(agm) == n - 1:
            break

    return agm, custo_total
    


In [ ]:
#prim
class Grafo:
    def __init__(self):
        self.adj = {}

    def adicionar_aresta(self, u, v, peso):
        if u not in self.adj: self.adj[u] = {}
        if v not in self.adj: self.adj[v] = {}

        self.adj[u][v] = peso
        self.adj[v][u] = peso

def converter_retorno_prim(arestas_prim, matriz_distancias):
    arestas_convertidas = []

    for aresta in arestas_prim:
        # Separa a string "0-1"
        origem, destino = map(int, aresta.split("-"))

        # Obtém o peso da aresta na matriz de distâncias
        peso = matriz_distancias[origem][destino]

        # Converte os vértices de 0...n-1 para 1...n
        arestas_convertidas.append(
            (peso, origem + 1, destino + 1)
        )

    return arestas_convertidas


def algoritmo_prim(grafo, raiz):
    chaves = {v: float('inf') for v in grafo.adj}
    chaves[raiz] = 0
    
    pais = {v: None for v in grafo.adj}
    
    fila_T = [(0, raiz)]
    visitados = set()
    agm = []
    peso_total = 0

    while fila_T:
        peso_u, u = heapq.heappop(fila_T)
        
        if u in visitados:
            continue
            
        visitados.add(u)
        
        if pais[u] is not None:
            agm.append(f"{pais[u]}-{u}")
            peso_total += peso_u

        for s, peso_us in grafo.adj[u].items():
            if s not in visitados and peso_us < chaves[s]:
                chaves[s] = peso_us
                pais[s] = u
                heapq.heappush(fila_T, (peso_us, s))
                
    return agm, peso_total


In [ ]:
instancia = 'instancias/tsp10.tsp'
coords = ler_tsp_arquivo(instancia)
matriz = criar_matriz_distancias(coords)

if matriz:
    V = len(matriz)
    g_numerico = Grafo()
    
    for i in range(V):
        for j in range(i + 1, V): 
            peso = matriz[i][j]
            if peso > 0: 
                g_numerico.adicionar_aresta(i, j, peso)


agm_prim, custo_total_prim = algoritmo_prim(g_numerico, raiz=0)
agm_prim_convertida = converter_retorno_prim( agm_prim, matriz)

agm_kruskal, custo_total_kruskal = kruskal_agm(matriz)


plotar_agm(coords,agm_kruskal,"Agm algoritmo de Kruskal", custo_total_kruskal)


plotar_agm(coords, agm_prim_convertida, "Árvore Geradora Mínima - Prim", custo_total_prim)

